# Day 11：文件操作与标准库（RAG 预处理基石）—— `pathlib`、`re`、`csv`、`json`、`datetime`

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 11 天！我是 Tech Lead 张工。在构建 RAG（检索增强生成）企业知识库之前，工程师必须先解决**脏乱差文档的扫描、清洗、词频统计与报告归档**。今天我们全程不依赖第三方库，只用 Python 标准库组合拳，完成 `DocScanner` 知识库文档扫描引擎！

## 1. 第一步：使用 `pathlib.Path` 构造离线知识库并递归过滤 `.txt` / `.md`

In [1]:
import tempfile
from pathlib import Path

tmp_obj = tempfile.TemporaryDirectory()
kb_dir = Path(tmp_obj.name) / "knowledge_base"
(kb_dir / "architecture").mkdir(parents=True, exist_ok=True)

(kb_dir / "rag_intro.md").write_text(
    "# 智枢平台 RAG 架构简介\n"
    "RAG（检索增强生成）结合了向量检索与大语言模型。\n"
    "在构建 RAG 管线时，文档切片与 Embedding 是核心环节。\n"
    "联系人手机：13812345678，邮箱：chenmo@xingchen.ai\n",
    encoding="utf-8",
)
(kb_dir / "architecture" / "agent_design.txt").write_text(
    "ZhiShu Agent 平台采用多智能体协作架构。\n"
    "Agent 通过 Tool 调用检索 RAG 知识库，提升问答准确率。\n",
    encoding="utf-8",
)
(kb_dir / "ignore_me.py").write_text("print('RAG')", encoding="utf-8")

target_files = sorted(
    (
        p for p in kb_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".txt", ".md"}
    ),
    key=lambda p: p.relative_to(kb_dir).as_posix(),
)
print("发现有效文本文档:", [p.relative_to(kb_dir).as_posix() for p in target_files])

发现有效文本文档: ['architecture/agent_design.txt', 'rag_intro.md']


## 2. 第二步：使用 `re` 正则表达式统计词频与脱敏隐私数据（PII）

In [2]:
import re

keywords = ["RAG", "Agent", "Embedding"]
patterns = {kw: re.compile(re.escape(kw), re.IGNORECASE) for kw in keywords}

for fp in target_files:
    text = fp.read_text(encoding="utf-8")
    counts = {kw: len(pat.findall(text)) for kw, pat in patterns.items()}
    print(f"{fp.name} -> 词频: {counts}")

# 实战演练：RAG 入库前对手机号与邮箱脱敏
phone_re = re.compile(r"(?<!\d)(1[3-9]\d)\d{4}(\d{4})(?!\d)")
email_re = re.compile(r"\b([A-Za-z0-9])[A-Za-z0-9._%+-]*(@[A-Za-z0-9.-]+\.[A-Za-z]{2,})\b")
raw_text = (kb_dir / "rag_intro.md").read_text(encoding="utf-8")
masked_text = email_re.sub(r"\1***\2", phone_re.sub(r"\1****\2", raw_text))
print("\n脱敏后末行:", masked_text.splitlines()[-1])

agent_design.txt -> 词频: {'RAG': 1, 'Agent': 2, 'Embedding': 0}
rag_intro.md -> 词频: {'RAG': 3, 'Agent': 0, 'Embedding': 1}

脱敏后末行: 联系人手机：138****5678，邮箱：c***@xingchen.ai


## 3. 第三步：结合 `datetime`、`csv` 与 `json` 导出双格式审计报告

In [3]:
import csv
import json
from datetime import datetime, timezone

records = []
for fp in target_files:
    text = fp.read_text(encoding="utf-8")
    stat = fp.stat()
    counts = {kw: len(pat.findall(text)) for kw, pat in patterns.items()}
    records.append({
        "file_name": fp.name,
        "relative_path": fp.relative_to(kb_dir).as_posix(),
        "size_bytes": stat.st_size,
        "modified_at": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
        "total_hits": sum(counts.values()),
        "keyword_counts": counts,
    })

out_dir = Path(tmp_obj.name) / "reports"
out_dir.mkdir(parents=True, exist_ok=True)

json_path = out_dir / "report.json"
payload = {
    "scanned_at": datetime.now(timezone.utc).isoformat(),
    "total_files": len(records),
    "files": records,
}
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

csv_path = out_dir / "report.csv"
fieldnames = ["file_name", "relative_path", "size_bytes", "modified_at", "total_hits", *[f"kw:{k}" for k in keywords]]
with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in records:
        row = {k: r[k] for k in ["file_name", "relative_path", "size_bytes", "modified_at", "total_hits"]}
        for kw in keywords:
            row[f"kw:{kw}"] = r["keyword_counts"][kw]
        writer.writerow(row)

print("JSON 报告预览:\n" + json_path.read_text(encoding="utf-8"))
print("CSV 报告预览:\n" + csv_path.read_text(encoding="utf-8"))
tmp_obj.cleanup()

JSON 报告预览:
{
  "scanned_at": "2026-11-13T05:38:21.324928+00:00",
  "total_files": 2,
  "files": [
    {
      "file_name": "agent_design.txt",
      "relative_path": "architecture/agent_design.txt",
      "size_bytes": 125,
      "modified_at": "2026-11-13T05:38:21.303052+00:00",
      "total_hits": 3,
      "keyword_counts": {
        "RAG": 1,
        "Agent": 2,
        "Embedding": 0
      }
    },
    {
      "file_name": "rag_intro.md",
      "relative_path": "rag_intro.md",
      "size_bytes": 233,
      "modified_at": "2026-11-13T05:38:21.303052+00:00",
      "total_hits": 4,
      "keyword_counts": {
        "RAG": 3,
        "Agent": 0,
        "Embedding": 1
      }
    }
  ]
}
CSV 报告预览:
file_name,relative_path,size_bytes,modified_at,total_hits,kw:RAG,kw:Agent,kw:Embedding
agent_design.txt,architecture/agent_design.txt,125,2026-11-13T05:38:21.303052+00:00,3,1,2,0
rag_intro.md,rag_intro.md,233,2026-11-13T05:38:21.303052+00:00,4,3,0,1

